# Anatomical axis PFC dataset figures

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Match atlas annotations

In [ ]:
# Qian PFC: ONE CELL FOR ALL THREE ANATOMICAL VALIDATION FIGURES.
# A: COMPACT GW20 Tissue 3 learned-z heatmap vs manual layer annotations.
# B: compact synthetic-style absolute Spearman bar chart, all 9 tissues.
# C: compact synthetic-style signed Spearman bar chart, all 9 tissues.
# Uses frozen Stage-1a native coordinates. NO retraining or annotation-based flips.
# Outputs all panels together in a single execution with PDF, SVG and PNG.
# Same visual comparison as near the bottom of Anatomical coordinate(1).ipynb.
# Run on the Skipjack CPU node containing the frozen model and atlas.

from pathlib import Path
import os, json, hashlib
import numpy as np
import pandas as pd
import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import Normalize, to_rgb
from scipy.spatial import cKDTree
from scipy.stats import spearmanr
from IPython.display import display

PROJECT = Path(os.environ.get('STP_PROJECT',
    os.environ["STP_PROJECT"]))
RUN = Path(os.environ.get('QIAN_SPATIAL_TI_NEW_RUN',
    str(PROJECT / 'experiments/qian_pfc_spatial_ti_all9_synthetic_defaults_v1/CPU_FULL')))
ATLAS = Path(os.environ.get('QIAN_INTEGRATED_ATLAS',
    str(Path(os.environ["QIAN_PFC_DATA_ROOT"]) / "merscope_integrated_855.h5ad")))
OUT = Path(os.environ.get('QIAN_ANATOMICAL_FIGURES_OUT',
    str(PROJECT / 'paper_figures/qian_all9_new_cpu_manual_annotated_GRAY_MASKED_compact_v3')))
DISPLAY_SECTION = 4          # fixed GW20 Tissue 3 (FB080/F2b), not score-selected
PLOT_POINTS = 240_000       # deterministic subsample of UNANNOTATED background only
MATCH_RADIUS = 2.0         # × median 5th nonself nearest-neighbor spacing
DPI = 400
SEED = 20261027
COMMIT = '76bb952a841c3671ec35161149e780167fd9b7b1'

SECTIONS = [
    (0, 'GW15', 'UMB1117', 'F1a', 548567),
    (1, 'GW15', 'UMB1117', 'F1b', 410492),
    (2, 'GW20', 'FB080', 'F1', 665204),
    (3, 'GW20', 'FB080', 'F2a', 596754),
    (4, 'GW20', 'FB080', 'F2b', 322648),
    (5, 'GW20', 'FB121', 'F1', 553738),
    (6, 'GW22', 'FB123', 'F1', 508417),
    (7, 'GW22', 'FB123', 'F2', 439986),
    (8, 'GW22', 'FB123', 'F3', 333113),
]

# Publication tissue names: ordinal numbering within each gestational week.
TISSUE_LABELS = {}
per_age_counts = {}
for sid, gw, sample, region, n in SECTIONS:
    per_age_counts[gw] = per_age_counts.get(gw, 0) + 1
    TISSUE_LABELS[sid] = f'{gw} Tissue {per_age_counts[gw]}'
assert TISSUE_LABELS[DISPLAY_SECTION] == 'GW20 Tissue 3'

LAYERS = ['vz','isvz','osvz','iz','sp','l6','l5','l4','l3','l2','mz']
LAYER_NAMES = ['VZ','iSVZ','oSVZ','IZ','SP','L6','L5','L4','L3','L2','MZ']
LAYER_COLORS = ['#2477AF','#9EC1E5','#FF8B27','#279C3C','#DE3435',
                '#9760C4','#8C564B','#E68CC6','#707070','#B6B51C','#20B9B9']
BLUE, RED, GRID = '#4C78A8', '#D62728', '#C9CDD2'

Z_FILE = RUN / 'tables/stage1a_all_cells.csv.gz'  # ALWAYS written by Stage-1a
PANEL = RUN / 'tables/panel_df.csv'
MODEL = RUN / 'model_final.pt'
FIT = RUN / 'FIT_VERIFIED.json'
LOCK = RUN / 'locked_config.json'

# 1. Validate newest run; deliberately NEVER fall back to October 13.
for tag, f in [('native learned-z',Z_FILE),('saved panel',PANEL),
               ('checkpoint',MODEL),('fit receipt',FIT),('training lock',LOCK),
               ('published atlas',ATLAS)]:
    if not f.is_file():
        raise FileNotFoundError(
            f'Missing {tag}: {f}\n'
            'Check the newest training run completed and set '
            'QIAN_SPATIAL_TI_NEW_RUN to its actual output folder. '
            'The code will NOT switch to an older model.'
        )
lock = json.loads(LOCK.read_text())
fit = json.loads(FIT.read_text())
assert fit.get('completed') is True
assert fit['method_version'] == 'stage1a_shared_logE_global_gauge_v1'
assert lock['active_commit'] == COMMIT
assert fit['n_sections'] == 9 and fit['n_cells'] == 4_378_919
assert fit['n_genes'] == 300 and fit['training_seed'] == 20260622
config = lock['training_config']
assert (config['epochs'], config['batch_size_per_slice'], config['seed']) == (1500,1024,20260622)
assert (config['coordinate_decoder'],config['gauge_mode'],config['use_library_size_offset']) == ('shared_auto','global',True)
# These markers must come from the same checkpoint.
h = hashlib.sha256()
with open(MODEL,'rb') as f:
    for buf in iter(lambda:f.read(4<<20), b''): h.update(buf)
assert h.hexdigest() == fit['checkpoint_sha256'], 'Model checkpoint does not match fit receipt'

panel = pd.read_csv(PANEL).sort_values('slice_id').reset_index(drop=True)
assert len(panel) == 9
observed = [(int(r.slice_id),str(r.gw),str(r.sample),str(r.region),int(r.n_full))
            for r in panel.itertuples()]
assert observed == SECTIONS, f'Unexpected panel, refusing to mislabel sections: {observed}'
OUT.mkdir(parents=True, exist_ok=True)
print('FROZEN MODEL:', MODEL, '\nNATIVE Z:', Z_FILE, '\nOUTPUT:', OUT)

# 2. Read native z and XY. No AnnData counts, GPU, or model dependencies.
# For memory efficiency, read only the 5 necessary columns in blocks.
arrays, indices = {s: [] for s in range(9)}, {s: [] for s in range(9)}
for block in pd.read_csv(
    Z_FILE, usecols=['slice_id','cell_index','x','y','z_stage1a'],
    dtype={'slice_id':'int16','cell_index':'int32',
           'x':'float32','y':'float32','z_stage1a':'float32'},
    chunksize=300_000
):
    for sid, b in block.groupby('slice_id',sort=False):
        sid=int(sid)
        if sid not in arrays: raise RuntimeError(f'Unknown section: {sid}')
        arrays[sid].append(b[['x','y','z_stage1a']].to_numpy(copy=True))
        indices[sid].append(b.cell_index.to_numpy(copy=True))
model = {}
for sid,gw,sample,region,n in SECTIONS:
    if not arrays[sid]: raise RuntimeError(f'Missing cells for {sid}')
    xyz = np.concatenate(arrays[sid])
    ids = np.concatenate(indices[sid])
    assert xyz.shape == (n,3), (sid,xyz.shape,n)
    assert np.array_equal(ids,np.arange(n)), f'Cell ordering mismatch in section {sid}'
    assert np.isfinite(xyz).all(), f'Nonfinite XY/z in section {sid}'
    model[sid] = xyz
    print(f'{sid} {gw} {sample}/{region}: {n:,} cells; native z [{xyz[:,2].min():.3f}, {xyz[:,2].max():.3f}]')
del arrays, indices

# 3. Published manual layer labels; nearest-XY transfer as in source notebook.
def decoded(x):
    return x.decode('utf-8') if isinstance(x,(bytes,np.bytes_)) else str(x)

def get_category(obs,name):
    g=obs[name]
    if not isinstance(g,h5py.Group) or not {'codes','categories'} <= set(g):
        raise RuntimeError(f'Expected categorical atlas obs/{name}')
    return np.asarray(g['codes'],np.int32), [decoded(v) for v in g['categories'][()]]

atlas = {}
with h5py.File(ATLAS,'r') as f:
    cs, samples = get_category(f['obs'],'sample')
    cr, regions = get_category(f['obs'],'region')
    cl, levels = get_category(f['obs'],'layer')
    levels = [v.strip().lower().replace(' ','') for v in levels]
    coords = f['obsm']['spatial']
    assert len(cs)==len(cr)==len(cl)==coords.shape[0]
    for sid,gw,sample,region,n in SECTIONS:
        assert sample in samples and region in regions, (sample,region)
        rows = np.flatnonzero((cs==samples.index(sample)) & (cr==regions.index(region)))
        rank = np.full(len(rows), -1, dtype=np.int16)
        for j,name in enumerate(LAYERS):
            if name in levels:
                rank[cl[rows] == levels.index(name)] = j
        chosen = np.flatnonzero(rank>=0)
        if len(chosen) == 0:
            atlas[sid] = (np.empty((0,2),np.float32), np.empty(0,np.int16), len(rows))
            continue
        xy = np.asarray(coords[rows[chosen],:2],np.float32)  # sorted HDF5 row indices
        atlas[sid] = (xy,rank[chosen],len(rows))

results, matches = [], {}
for sid,gw,sample,region,n in SECTIONS:
    xyz = model[sid]
    xy,rank,n_atlas = atlas[sid]
    tree = cKDTree(xyz[:,:2])
    rng=np.random.default_rng(SEED+sid)
    refs=rng.choice(n,min(5000,n),replace=False)
    k=6  # includes self; index 5 is fifth nonself neighbor
    spacing=float(np.median(tree.query(xyz[refs,:2],k=k,workers=-1)[0][:,5]))
    if spacing<=0 or not np.isfinite(spacing): raise RuntimeError(f'Invalid spacing: {sid}')
    if len(xy):
        dist,ix = tree.query(xy,k=1,workers=-1)
        keep = np.isfinite(dist)&(dist <= MATCH_RADIUS*spacing)
        r = float(spearmanr(xyz[ix[keep],2],rank[keep]).statistic) if keep.sum()>20 else np.nan
    else:
        dist=np.empty(0); ix=np.empty(0,dtype=int); keep=np.empty(0,dtype=bool); r=np.nan
    matches[sid] = (xy,rank,ix,keep)
    results.append(dict(slice_id=sid,tissue_label=TISSUE_LABELS[sid],gw=gw,sample=sample,region=region,
        n_model=n,n_atlas=n_atlas,n_labeled=len(xy),n_matched=int(keep.sum()),
        match_fraction=float(keep.mean()) if len(keep) else np.nan,
        signed_spearman=r,absolute_spearman=abs(r)))
metrics=pd.DataFrame(results)
metrics.to_csv(OUT/'all9_spearman_native_z_vs_published_layers.csv',index=False)
print('\nNATIVE anatomical ordering (no layer-informed sign changes):')
display(metrics[['tissue_label','slice_id','gw','sample','region','n_labeled','n_matched',
                 'signed_spearman','absolute_spearman']].round(4))

## Render Figure 5A

In [ ]:
# Run AFTER the frozen all-nine PFC anatomical validation cell.
# Inputs already in memory: model, matches, metrics, LAYERS, LAYER_COLORS, SEED, OUT.
# No refitting, no new matching, no label-informed z flips or changes to Spearman.
# GW15 / GW20 / GW22 rows, Tissue 1..4 columns; unused entries remain empty.
# For every tissue: learned common z (left) and manual annotations (right).
# White curves are FOUR EVENLY SPACED LEARNED-z ISOCONTOURS in each tissue,
# NOT manual boundaries; contour locations and levels are independent of labels.
# Spatially matched annotation overlap is vivid; unmatched tissue is gray-muted.
# Single footer row with font-measured spacing for label, colorbar and 11 layers.

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib.colors import Normalize, to_rgb
from matplotlib.patches import Patch
from matplotlib import font_manager
from scipy.spatial import cKDTree

# ------------------------------------------------------------
# 0. Validate frozen inputs. No scientific results are changed.
# ------------------------------------------------------------
REQUIRED = ('model', 'matches', 'metrics', 'LAYERS', 'LAYER_COLORS', 'SEED', 'OUT')
for name in REQUIRED:
    if name not in globals():
        raise RuntimeError(f"Missing '{name}'. Run the all-nine PFC anatomical validation cell first.")
assert isinstance(metrics, pd.DataFrame)
assert {'slice_id', 'signed_spearman'} <= set(metrics.columns)
assert len(LAYERS) == len(LAYER_COLORS) == 11
ROW_IDS = [(0, 1), (2, 3, 4, 5), (6, 7, 8)]
ROW_LABELS = ('GW15', 'GW20', 'GW22')
COLUMN_LABELS = ('Tissue 1', 'Tissue 2', 'Tissue 3', 'Tissue 4')
SIDS = tuple(range(9))
assert all(s in model and s in matches for s in SIDS)
LAYER_DISPLAY = list(globals().get(
    'LAYER_NAMES', ['VZ', 'iSVZ', 'oSVZ', 'IZ', 'SP', 'L6', 'L5', 'L4', 'L3', 'L2', 'MZ']))
assert len(LAYER_DISPLAY) == 11

rho_indexed = metrics.set_index('slice_id')
assert rho_indexed.index.is_unique and set(SIDS) <= set(rho_indexed.index)
RHO = {s: float(rho_indexed.loc[s, 'signed_spearman']) for s in SIDS}
assert all(np.isfinite(v) and -1.000001 <= v <= 1.000001 for v in RHO.values())

OUT = Path(OUT)
OUT.mkdir(parents=True, exist_ok=True)
GRAY = '#AEB4BD'
GRAY_RGB = np.asarray(to_rgb(GRAY), dtype=np.float32)
INK = '#25343D'
CMAP = plt.get_cmap('viridis')
fonts = {entry.name for entry in font_manager.fontManager.ttflist}
PAPER_FONT = next((x for x in ('Arial', 'Arimo', 'Liberation Sans',
                               'Nimbus Sans', 'DejaVu Sans') if x in fonts), 'DejaVu Sans')
mpl.rcParams.update({
    'font.family': PAPER_FONT,
    'font.size': 10.4,
    'axes.titlesize': 12.4,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11.2,
    'xtick.labelsize': 10.0,
    'ytick.labelsize': 10.0,
    'text.color': INK,
    'axes.labelcolor': INK,
    'xtick.color': INK,
    'ytick.color': INK,
    'axes.edgecolor': INK,
    'axes.linewidth': .75,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white',
})

ZNORM = Normalize(vmin=min(float(model[s][:, 2].min()) for s in SIDS),
                  vmax=max(float(model[s][:, 2].max()) for s in SIDS))
assert ZNORM.vmax > ZNORM.vmin

# ------------------------------------------------------------
# 1. Publication dimensions: preserve the LARGER latest figure.
# ------------------------------------------------------------
COL_W = 2.18
COL_GAP = .16
INNER_GAP = .08
LEFT_MARGIN = .68
RIGHT_MARGIN = .20
MAX_IMAGE_H = 1.26
TOP_HEADROOM = .44
SCORE_HEADER_H = .22
ROW_GAP = .16
FOOTER_H = .86
FIG_W = LEFT_MARGIN + 4 * COL_W + 3 * COL_GAP + RIGHT_MARGIN  # 10.08 in

# Point limits are visualization-only, never used in scoring or level selection.
MAX_MUTED_POINTS = 85_000
MAX_VIVID_POINTS = 170_000
MAX_MANUAL_PER_LAYER = 120_000
MAX_CONTOUR_POINTS = 17_000
N_CONTOURS = 4
CONTOUR_LINEWIDTH_PT = 1.45
PNG_DPI = 450

GEOM = {}
for sid in SIDS:
    xyz = np.asarray(model[sid])
    assert xyz.ndim == 2 and xyz.shape[1] == 3 and np.isfinite(xyz).all(), sid
    px = max(float(np.ptp(xyz[:, 0])) * .025, 1e-6)
    py = max(float(np.ptp(xyz[:, 1])) * .025, 1e-6)
    xmin, xmax = float(xyz[:, 0].min()-px), float(xyz[:, 0].max()+px)
    ymin, ymax = float(xyz[:, 1].min()-py), float(xyz[:, 1].max()+py)
    aspect = (xmax-xmin) / (ymax-ymin)
    image_h = min(MAX_IMAGE_H, (COL_W-INNER_GAP)/(2*aspect))
    GEOM[sid] = dict(xlim=(xmin, xmax), ylim=(ymax, ymin),
                     image_h=image_h, image_w=image_h*aspect)

ROW_HEIGHTS = [max(GEOM[s]['image_h'] for s in row) for row in ROW_IDS]
row_top = []
cursor = TOP_HEADROOM
for i, hh in enumerate(ROW_HEIGHTS):
    row_top.append(cursor)
    cursor += SCORE_HEADER_H + hh + (ROW_GAP if i < 2 else 0.)
CONTENT_END = cursor
FIG_H = CONTENT_END + FOOTER_H
fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor='white')

def from_top(top_inches):
    return (FIG_H - top_inches) / FIG_H

def sample_idx(idx, count, random_seed):
    idx = np.asarray(idx, dtype=np.int64)
    if len(idx) <= count:
        return idx
    rng = np.random.default_rng(int(random_seed))
    return np.sort(rng.choice(idx, count, replace=False))

# Triangulate a bounded subset, to avoid Delaunay on ~500k cells per tissue.
# Remove long-edge triangles, which otherwise bridge holes/concavities.
def local_tri_for_z(xyz, sid):
    n = len(xyz)
    idx = sample_idx(np.arange(n), MAX_CONTOUR_POINTS, int(SEED)+40_000+sid)
    xx, yy = xyz[idx, 0], xyz[idx, 1]
    tri = mtri.Triangulation(xx, yy)
    tris = tri.triangles
    if len(tris) < 1:
        return None
    points = np.column_stack((xx, yy))
    dist = cKDTree(points).query(points, k=min(6, len(points)), workers=-1)[0]
    local = dist[:, -1] if dist.ndim == 2 else dist
    positive = local[np.isfinite(local) & (local > 0)]
    if len(positive) == 0:
        return None
    # Avoid anomalous duplicate-coordinate nearest-neighbor distances.
    lo, hi = np.quantile(positive, [.05, .95])
    local = np.clip(local, max(lo, 1e-9), max(hi, lo, 1e-9))
    p0, p1, p2 = points[tris[:, 0]], points[tris[:, 1]], points[tris[:, 2]]
    e01 = np.linalg.norm(p0-p1, axis=1)
    e12 = np.linalg.norm(p1-p2, axis=1)
    e20 = np.linalg.norm(p2-p0, axis=1)
    mask = ((e01 > 3.5*np.maximum(local[tris[:,0]], local[tris[:,1]])) |
            (e12 > 3.5*np.maximum(local[tris[:,1]], local[tris[:,2]])) |
            (e20 > 3.5*np.maximum(local[tris[:,2]], local[tris[:,0]])) |
            (np.maximum.reduce((e01,e12,e20)) > 6.0*np.median(positive)))
    tri.set_mask(mask)
    return tri, np.asarray(xyz[idx, 2], dtype=float)

# ------------------------------------------------------------
# 2. Column and gestational-week labels
# ------------------------------------------------------------
for c, label in enumerate(COLUMN_LABELS):
    center = LEFT_MARGIN + c*(COL_W+COL_GAP) + COL_W/2
    fig.text(center/FIG_W, from_top(.13), label, ha='center', va='top',
             fontsize=12.6, fontweight='bold')

contour_levels_by_tissue = {}
for row_num, (gw, group) in enumerate(zip(ROW_LABELS, ROW_IDS)):
    row_imagetop = row_top[row_num] + SCORE_HEADER_H
    fig.text(.18/FIG_W, from_top(row_imagetop + ROW_HEIGHTS[row_num]/2),
             gw, rotation=90, ha='center', va='center',
             fontsize=12.3, fontweight='bold')

    for c, sid in enumerate(group):
        geom = GEOM[sid]
        iw, ih = geom['image_w'], geom['image_h']
        center = LEFT_MARGIN + c*(COL_W+COL_GAP) + COL_W/2
        left = center - (2*iw+INNER_GAP)/2
        img_top = row_imagetop + (ROW_HEIGHTS[row_num]-ih)/2
        bottom = FIG_H - img_top - ih

        fig.text(center/FIG_W, from_top(row_top[row_num]+.085),
                 rf'$\rho$ = {RHO[sid]:+.3f}', ha='center', va='center',
                 fontsize=11.2, fontweight='bold')

        axs = []
        for j in range(2):
            ax = fig.add_axes([(left+j*(iw+INNER_GAP))/FIG_W,
                               bottom/FIG_H, iw/FIG_W, ih/FIG_H])
            ax.set_xlim(*geom['xlim'])
            ax.set_ylim(*geom['ylim'])  # published atlas orientation: y inverted
            ax.set_aspect('equal', adjustable='box')
            ax.set_anchor('C')
            ax.axis('off')
            axs.append(ax)

        xyz = np.asarray(model[sid])
        n = len(xyz)
        atlas_xy, atlas_rank, nn_idx, accepted = matches[sid]
        accepted = np.asarray(accepted, dtype=bool)
        foreground = np.unique(np.asarray(nn_idx, dtype=np.int64)[accepted])
        assert (foreground >= 0).all() and (foreground < n).all(), sid
        is_annotated = np.zeros(n, dtype=bool)
        is_annotated[foreground] = True
        background = np.flatnonzero(~is_annotated)
        background = sample_idx(background, MAX_MUTED_POINTS, int(SEED)+10_000+sid)
        vivid = sample_idx(foreground, MAX_VIVID_POINTS, int(SEED)+20_000+sid)

        # Left: annotation-overlap cells in vivid native z; otherwise gray-muted.
        if len(background):
            muted = CMAP(ZNORM(xyz[background, 2])).astype(np.float32)
            muted[:, :3] = .65*muted[:, :3] + .35*GRAY_RGB
            muted[:, 3] = .67
            axs[0].scatter(xyz[background, 0], xyz[background, 1],
                           c=muted, s=.40, edgecolors='none',
                           rasterized=True, zorder=1)
        if len(vivid):
            axs[0].scatter(xyz[vivid, 0], xyz[vivid, 1],
                           c=xyz[vivid, 2], cmap=CMAP, norm=ZNORM,
                           s=.76, edgecolors='none', rasterized=True, zorder=3)

        # Four evenly spaced, tissue-specific z values. No manual labels used.
        z = xyz[:, 2]
        low, high = np.quantile(z, [.16, .84])
        contour_levels = np.linspace(float(low), float(high), N_CONTOURS)
        contour_levels_by_tissue[sid] = contour_levels.tolist()
        tri_result = local_tri_for_z(xyz, sid)
        if tri_result is not None and high > low:
            tri, zz = tri_result
            # Matplotlib >= 3.8 compatible; no TriContourSet.collections call.
            axs[0].tricontour(tri, zz, levels=contour_levels,
                              colors='white', linewidths=CONTOUR_LINEWIDTH_PT,
                              alpha=.98, zorder=4)

        # Right: exactly the same spatial annotation overlap in native layer hues.
        if len(background):
            axs[1].scatter(xyz[background, 0], xyz[background, 1],
                           c=GRAY, alpha=.40, s=.40,
                           edgecolors='none', rasterized=True, zorder=1)
        xy = np.asarray(atlas_xy)[accepted]
        rank = np.asarray(atlas_rank, dtype=int)[accepted]
        for layer in range(len(LAYERS)):
            ids = np.flatnonzero(rank == layer)
            ids = sample_idx(ids, MAX_MANUAL_PER_LAYER,
                             int(SEED)+30_000+37*sid+layer)
            if len(ids):
                axs[1].scatter(xy[ids, 0], xy[ids, 1], color=LAYER_COLORS[layer],
                               s=.72, edgecolors='none', rasterized=True, zorder=3)
        if not len(xy):
            axs[1].text(.5, .5, 'No labels', transform=axs[1].transAxes,
                        ha='center', va='center', fontsize=8)

# ------------------------------------------------------------
# 3. One-row footer: measured text and collision-free positions
# ------------------------------------------------------------
# Keep the 10.08-inch canvas and all 11 anatomical labels on the same row.
# Measure the text using the installed font before setting the colorbar x.
# This avoids label/bar collisions when font metrics differ across nodes.
FOOTER_CENTER_Y = .47  # inches from bottom of canvas
COORD_LABEL_LEFT = .47  # inches from left of canvas
COORD_LABEL = fig.text(
    COORD_LABEL_LEFT/FIG_W, FOOTER_CENTER_Y/FIG_H,
    'Common coordinate', ha='left', va='center',
    fontsize=11.4, fontweight='normal'  # reliable weight on all fonts
)
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
label_right = COORD_LABEL.get_window_extent(renderer).x1 / fig.dpi

# Reserve 0.12 in after the label and 0.17 in before the legend.
LEGEND_LEFT = 4.32
BAR_LEFT = max(2.06, label_right + .12)
BAR_RIGHT = LEGEND_LEFT - .17
BAR_W, BAR_H = BAR_RIGHT - BAR_LEFT, .11
if BAR_W < 1.10:
    raise RuntimeError(
        f'Footer font too wide to fit one row: label ends at {label_right:.3f} in; '
        f'colorbar has only {BAR_W:.3f} in. Reduce fontsize or enlarge footer.')

cax = fig.add_axes([BAR_LEFT/FIG_W, (FOOTER_CENTER_Y-BAR_H/2)/FIG_H,
                    BAR_W/FIG_W, BAR_H/FIG_H])
sm = mpl.cm.ScalarMappable(norm=ZNORM, cmap=CMAP)
sm.set_array([])
cb = fig.colorbar(sm, cax=cax, orientation='horizontal')
cb.ax.tick_params(labelsize=9.5, length=2.4, pad=1.6, width=.65)
cb.outline.set_linewidth(.65)

handles = [Patch(facecolor=LAYER_COLORS[k], edgecolor='none', label=LAYER_DISPLAY[k])
           for k in range(len(LAYERS))]
legend = fig.legend(handles=handles, loc='center left',
                    bbox_to_anchor=(LEGEND_LEFT/FIG_W, FOOTER_CENTER_Y/FIG_H),
                    ncol=len(handles), frameon=False, fontsize=9.7,
                    handlelength=.80, handleheight=.75,
                    handletextpad=.24, columnspacing=.48,
                    borderaxespad=0, labelspacing=0)

# Only the legend font adjusts if the installed font is unusually wide.
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
right_edge_in = legend.get_window_extent(renderer).x1 / fig.dpi
if right_edge_in > FIG_W - .12:
    legend.remove()
    legend = fig.legend(handles=handles, loc='center left',
                        bbox_to_anchor=(LEGEND_LEFT/FIG_W, FOOTER_CENTER_Y/FIG_H),
                        ncol=len(handles), frameon=False, fontsize=9.0,
                        handlelength=.74, handleheight=.75,
                        handletextpad=.18, columnspacing=.34,
                        borderaxespad=0, labelspacing=0)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    right_edge_in = legend.get_window_extent(renderer).x1 / fig.dpi
assert right_edge_in <= FIG_W - .04, (right_edge_in, FIG_W)

# Rendered, not estimated, element bounds. These should now always be clear.
assert label_right + .10 <= BAR_LEFT, (label_right, BAR_LEFT)
assert BAR_LEFT + BAR_W + .12 <= LEGEND_LEFT, (BAR_LEFT, BAR_W, LEGEND_LEFT)
print(f'Footer clearances: label→bar {BAR_LEFT-label_right:.3f} in; '
      f'bar→legend {LEGEND_LEFT-BAR_LEFT-BAR_W:.3f} in')

# ------------------------------------------------------------
# 4. Export fixed-size SVG/PDF/PNG; report provenance.
# ------------------------------------------------------------
full_bounds = np.asarray(fig.get_tightbbox(renderer).extents, float)
print('Publication canvas:', round(FIG_W, 3), 'x', round(FIG_H, 3), 'in')
print('Rendered bounds:', np.round(full_bounds, 3))
assert (full_bounds[0] >= -.015 and full_bounds[1] >= -.015 and
        full_bounds[2] <= FIG_W+.015 and full_bounds[3] <= FIG_H+.015), full_bounds

stem = OUT / 'all9_native_z_vs_manual_even_z_contours_one_row_footer_PUBLICATION'
for ext in ('pdf', 'svg', 'png'):
    path = stem.with_suffix('.'+ext)
    fig.savefig(path, facecolor='white', bbox_inches=None, pad_inches=0,
                dpi=PNG_DPI if ext == 'png' else None)
    print('SAVED:', path)

receipt = {
    'figure_size_inches': [round(FIG_W, 4), round(FIG_H, 4)],
    'native_z_unchanged': True,
    'signed_spearman_reused': True,
    'foreground': 'accepted nearest matched model cells for published atlas annotations',
    'background': 'other tissue cells, deterministic display-only subsample',
    'contours': 'four tissue-specific evenly spaced learned-z isocontours; no atlas labels used',
    'contour_linewidth_points': CONTOUR_LINEWIDTH_PT,
    'contour_levels': {str(k): v for k, v in contour_levels_by_tissue.items()},
    'footer': 'single row: common coordinate label, horizontal z colorbar, eleven-layer legend',
}
(stem.parent / (stem.name+'_receipt.json')).write_text(json.dumps(receipt, indent=2)+'\n')
plt.show()
plt.close(fig)
print('Signed Spearman reused unchanged:', RHO)

## Fit gene-wise Poisson models and score ARI

In [ ]:
# Qian PFC | Belayer-style gene-wise fixed-K Poisson segmentation | ARI
# Each gene in each tissue gets INDEPENDENT expression-derived breakpoints.
# K = number of distinct manually annotated anatomical layers in that tissue.
# No elbow; manual labels determine K and post-fit ARI, NOT breakpoint locations.
# ARI is calculated on unique atlas-matched CELLS exactly via binned counts.
# No retraining of Zoetrope. No GLM-PCA. Runs on CPU.

from pathlib import Path
import os, json, time, hashlib
import numpy as np
import pandas as pd
import h5py
from scipy.spatial import cKDTree
from scipy.ndimage import gaussian_filter1d
from matplotlib import pyplot as plt
from matplotlib import patches as mpatches
from matplotlib import lines as mlines
from matplotlib import ticker as mticker
import matplotlib as mpl

# -------------------- CONFIG --------------------
PROJECT = Path(os.environ.get('STP_PROJECT',
    os.environ["STP_PROJECT"]))
RUN = Path(os.environ.get('QIAN_SPATIAL_TI_NEW_RUN',
    str(PROJECT/'experiments/qian_pfc_spatial_ti_all9_synthetic_defaults_v1/CPU_FULL')))
POISSON_SOURCE = Path(os.environ.get('STP_PWPOISSON_DIR',
    str(PROJECT/'paper_figures/pfc_z_joint_poisson_dp_9sections_v1')))
ATLAS = Path(os.environ.get('QIAN_INTEGRATED_ATLAS',
    str(Path(os.environ["QIAN_PFC_DATA_ROOT"]) / "merscope_integrated_855.h5ad")))
Z_TABLE = RUN/'tables/stage1a_all_cells.csv.gz'
OUT = PROJECT/'paper_figures/pfc_gene_specific_fixedK_ARI_all9_v1'
OUT.mkdir(parents=True, exist_ok=True)

MIN_BINS_PER_SEGMENT = 5  # original direct-Poisson minimum segment size
MATCH_RADIUS = 2.0        # atlas->Stage1 NN radius, in median fifth-NN spacings
SEED = 20261030
SCALE = 1e4               # expression per 10,000 assayed-panel UMIs
DPI = 450
REUSE_INTERVAL_CACHE = True
ONLY_SIDS = None          # set e.g. (4,) to smoke-test GW20 Tissue 3

SECTIONS = [
    (0,'GW15','UMB1117','F1a',548567),
    (1,'GW15','UMB1117','F1b',410492),
    (2,'GW20','FB080','F1',665204),
    (3,'GW20','FB080','F2a',596754),
    (4,'GW20','FB080','F2b',322648),
    (5,'GW20','FB121','F1',553738),
    (6,'GW22','FB123','F1',508417),
    (7,'GW22','FB123','F2',439986),
    (8,'GW22','FB123','F3',333113),
]
CODES = ('vz','isvz','osvz','iz','sp','l6','l5','l4','l3','l2','mz')
NAMES = ('VZ','iSVZ','oSVZ','IZ','SP','L6','L5','L4','L3','L2','MZ')
COLORS = ('#2477AF','#9EC1E5','#FF8B27','#279C3C','#DE3435',
          '#9760C4','#8C564B','#E68CC6','#707070','#B6B51C','#20B9B9')

INK  = '#26343D'
GRID = '#CFD5DB'
GRAY = '#9AA3AE'

SELECTED = [s for s in SECTIONS if ONLY_SIDS is None or s[0] in ONLY_SIDS]
if not SELECTED:
    raise ValueError('No sections selected')
for f in (Z_TABLE, ATLAS):
    if not f.is_file():
        raise FileNotFoundError(f)

# -------------------- STYLE GUIDE UPDATE --------------------
mpl.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 9.2,
    'axes.titlesize': 10.4,
    'axes.titleweight': 'bold',
    'axes.labelsize': 9.8,
    'xtick.labelsize': 8.8,
    'ytick.labelsize': 8.8,
    'axes.linewidth': 0.72,
    'axes.edgecolor': INK,
    'axes.labelcolor': INK,
    'xtick.color': INK,
    'ytick.color': INK,
    'text.color': INK,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
    'savefig.facecolor': 'white',
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

def decode(v):
    return v.decode('utf8') if isinstance(v, (bytes, np.bytes_)) else str(v)

def categoricals(group):
    if not isinstance(group, h5py.Group) or not {'codes', 'categories'} <= set(group):
        raise RuntimeError('Expected h5ad categorical group')
    return np.asarray(group['codes'], np.int32), [decode(x) for x in group['categories'][:]]

# -------------------- 1. Source inputs --------------------
D = {}
for sid, gw, sample, region, n in SELECTED:
    folder = POISSON_SOURCE / f'section_{sid:02d}_{gw}_{sample}_{region}'
    inp = folder/'binned_original_counts.npz'
    rank_path = folder/'gene_rankings_and_coefficients.csv'
    cell_bins_path = folder/'depth_bins_and_segments.csv'
    for f in (inp, rank_path, cell_bins_path):
        if not f.is_file():
            raise FileNotFoundError(f'Missing original (NON-elbow) Poisson source: {f}')
    with np.load(inp, allow_pickle=False) as zf:
        Y = np.asarray(zf['Y'], float)
        W = np.asarray(zf['W'], float)
        x = np.asarray(zf['center'], float)
        edges = np.asarray(zf['edges'], float)
        gene_names = np.asarray(zf['gene_names']).astype(str)
    oldrank = pd.read_csv(rank_path)
    oldgenes = oldrank['gene'].astype(str).to_numpy()
    keep = pd.Index(gene_names).get_indexer(oldgenes)
    assert len(keep) > 0 and (keep >= 0).all() and len(set(keep)) == len(keep)
    n_by_bin = pd.read_csv(cell_bins_path)['n_cells'].to_numpy(dtype=int)
    assert Y.shape == (len(x), len(gene_names)) and len(W) == len(x)
    assert len(edges) == len(x)+1 and (np.diff(edges) > 0).all() and (np.diff(x) > 0).all()
    assert np.isfinite(Y).all() and (Y >= 0).all() and (W > 0).all()
    assert np.allclose(Y.sum(1), W, atol=1e-5, rtol=1e-7)
    assert n_by_bin.sum() == n, f'sid={sid}: saved source cells differ from frozen panel'
    d = {
        'sid': sid, 'gw': gw, 'sample': sample, 'region': region, 'n': n,
        'x': x, 'edges': edges, 'W': W, 'Y': Y[:, keep],
        'names': gene_names[keep],
        'detection': dict(zip(oldrank.gene.astype(str), oldrank['detection_fraction']))
    }
    D[sid] = d
    print(f'{gw} {sample}/{region}: {len(x)} bins, {len(keep)} original filtered genes')

# -------------------- 2. Match atlas labels to native learned z --------------------
parts = {sid: [] for sid in D}
order = {sid: [] for sid in D}
for chunk in pd.read_csv(
    Z_TABLE,
    usecols=['slice_id', 'cell_index', 'x', 'y', 'z_stage1a'],
    dtype={'slice_id': 'int16', 'cell_index': 'int32', 'x': 'float32',
           'y': 'float32', 'z_stage1a': 'float32'},
    chunksize=350000
):
    for sid, sub in chunk.groupby('slice_id', sort=False):
        sid = int(sid)
        if sid in D:
            order[sid].append(sub.cell_index.to_numpy(dtype=np.int32, copy=True))
            parts[sid].append(sub[['x', 'y', 'z_stage1a']].to_numpy(np.float32, copy=True))

for sid in D:
    if not parts[sid]:
        raise RuntimeError(f'Missing frozen Stage1 slice {sid}')
    idx = np.concatenate(order[sid])
    xyz = np.concatenate(parts[sid])
    if len(idx) != D[sid]['n'] or not np.array_equal(idx, np.arange(len(idx))):
        raise RuntimeError(f'Frozen Stage1 row order/count mismatch: sid {sid}')
    D[sid]['xyz'] = xyz
parts.clear()
order.clear()

with h5py.File(ATLAS, 'r') as f:
    sc, samples = categoricals(f['obs']['sample'])
    rc, regions = categoricals(f['obs']['region'])
    lc, layers = categoricals(f['obs']['layer'])
    canonical = [a.strip().lower().replace(' ', '') for a in layers]
    layer_map = {j: CODES.index(v) for j, v in enumerate(canonical) if v in CODES}
    if len(layer_map) < 6:
        raise RuntimeError(f'Unexpected manual layer categories: {layers}')
    for sid, d in D.items():
        rows = np.flatnonzero((sc == samples.index(d['sample'])) &
                              (rc == regions.index(d['region'])))
        labels = np.full(len(rows), -1, np.int16)
        for source_label, layer in layer_map.items():
            labels[lc[rows] == source_label] = layer
        ok = labels >= 0
        atlas_xy = np.asarray(f['obsm']['spatial'][rows[ok], :2], np.float32)
        labels = labels[ok]
        src = d.pop('xyz')
        tree = cKDTree(src[:, :2])
        rng = np.random.default_rng(SEED + sid)
        probes = rng.choice(len(src), min(5000, len(src)), replace=False)
        spacing = float(np.median(tree.query(src[probes, :2], k=6, workers=-1)[0][:, 5]))
        dd, nearest = tree.query(atlas_xy, k=1, workers=-1)
        accepted = np.isfinite(dd) & (dd <= MATCH_RADIUS * spacing)
        if accepted.sum() < 100:
            raise RuntimeError(f'sid {sid}: too few atlas matches')
        ids = nearest[accepted]
        labs = labels[accepted]
        distances = dd[accepted]
        sort = np.lexsort((distances, ids))
        ids, labs = ids[sort], labs[sort]
        first = np.r_[True, ids[1:] != ids[:-1]]
        ids, labs = ids[first], labs[first]
        native_z = src[ids, 2].astype(float)
        B = len(d['x'])
        bin_id = np.clip(np.searchsorted(d['edges'], native_z, side='right') - 1, 0, B-1)
        present = np.unique(labs)
        K = len(present)
        if not (2 <= K <= len(CODES) and K * MIN_BINS_PER_SEGMENT <= B):
            raise RuntimeError(f'sid={sid} has K={K} zones and {B} bins; '
                               'cannot enforce fixed K and minimum segment width')
        ordered = sorted(present, key=lambda a: (np.median(native_z[labs == a]), int(a)))
        local = np.full(len(CODES), -1, dtype=int)
        local[np.asarray(ordered, dtype=int)] = np.arange(K)
        votes = np.zeros((B, K), dtype=np.int64)
        np.add.at(votes, (bin_id, local[labs]), 1)
        assert votes.sum() == len(ids)
        d.update(
            K=K,
            zone_ids=np.asarray(ordered, int),
            votes=votes,
            n_labeled_cells=int(len(ids))
        )
        print(f'  sid {sid}: K={K}, unique manually labeled cells={len(ids):,}, '
              f'layers={"/".join(NAMES[i] for i in ordered)}')
        del src, tree, atlas_xy

# -------------------- 3. Profiled direct-Poisson interval costs --------------------
def fit_poisson_interval(yy, w, x, return_coeff=False):
    yy = np.asarray(yy, float)
    w = np.asarray(w, float)
    x = np.asarray(x, float)
    n, G = yy.shape
    Y = yy.sum(0)
    T = x @ yy
    mx = float(w @ x / w.sum())
    xc = x - mx
    active = Y > 0
    beta = np.zeros(G, float)
    alpha = np.full(G, -np.inf, float)
    if np.any(active):
        target = T[active] / Y[active] - mx
        cap = min(10000., 55. / max(float(x[-1] - x[0]), 1e-6))
        lo = np.full(len(target), -cap)
        hi = np.full(len(target), cap)
        for _ in range(35):
            mid = (lo + hi) * 0.5
            e = np.exp(mid[:, None] * xc[None, :]) * w[None, :]
            means = (e @ xc) / e.sum(1)
            lo = np.where(means < target, mid, lo)
            hi = np.where(means >= target, mid, hi)
        bb = (lo + hi) * 0.5
        expo = np.exp(np.clip(bb[:, None] * xc[None, :], -65., 65.)) * w[None, :]
        norm = expo.sum(1)
        beta[active] = bb
        alpha[active] = np.log(Y[active]) - np.log(norm) - bb * mx
    cost = np.zeros(G, float)
    cost[active] = Y[active] * (1. - alpha[active]) - beta[active] * T[active]
    if return_coeff:
        return cost, alpha, beta
    return cost

def fingerprint(d):
    h = hashlib.sha256()
    for arr in (d['Y'], d['W'], d['x'], d['edges']):
        h.update(np.ascontiguousarray(arr).tobytes())
    h.update('\0'.join(map(str, d['names'])).encode('utf8'))
    h.update(f'Poisson35_minbins{MIN_BINS_PER_SEGMENT}_v1'.encode())
    return h.hexdigest()

def interval_costs(d):
    x = d['x']
    w = d['W']
    Y = d['Y']
    B, G = Y.shape
    path = OUT / f'section_{d["sid"]:02d}_gene_interval_costs_cache.npz'
    digest = fingerprint(d)
    if REUSE_INTERVAL_CACHE and path.is_file():
        with np.load(path, allow_pickle=False) as f:
            if str(f['digest']) == digest and f['cost'].shape == (B+1, B+1, G):
                print(f'  sid {d["sid"]}: reused interval cost cache')
                return f['cost'].astype(float)
    C = np.full((B+1, B+1, G), np.inf, dtype=float)
    m = MIN_BINS_PER_SEGMENT
    for i in range(B - m + 1):
        if i % 25 == 0:
            print(f'  sid {d["sid"]}: intervals starting at {i}/{B-m+1}', flush=True)
        for j in range(i + m, B + 1):
            C[i, j, :] = fit_poisson_interval(Y[i:j], w[i:j], x[i:j])
    if REUSE_INTERVAL_CACHE:
        np.savez_compressed(path, cost=C, digest=np.array(digest))
    return C

def gene_specific_dp(C, K, m):
    B = C.shape[0] - 1
    G = C.shape[2]
    dp = np.full((K+1, B+1, G), np.inf)
    back = np.full((K+1, B+1, G), -1, dtype=np.int32)
    dp[0, 0, :] = 0.
    genes = np.arange(G)
    for k in range(1, K+1):
        for j in range(k*m, B+1):
            ii = np.arange((k-1)*m, j-m+1)
            val = dp[k-1, ii, :] + C[ii, j, :]
            arg = np.argmin(val, axis=0)
            dp[k, j, :] = val[arg, genes]
            back[k, j, :] = ii[arg]
    if not np.all(np.isfinite(dp[K, B])):
        raise RuntimeError('Nonfinite gene-specific DP result')
    spans = []
    for g in range(G):
        j = B
        one = []
        for k in range(K, 0, -1):
            i = int(back[k, j, g])
            one.append((i, j))
            j = i
        one = one[::-1]
        if j != 0 or one[0][0] != 0 or one[-1][1] != B:
            raise AssertionError('Invalid DP reconstruction')
        spans.append(one)
    return spans, dp[K, B].copy()

# -------------------- 4. ARI on matched cells --------------------
def ari_from_contingency(c):
    c = np.asarray(c, np.float64)
    n = float(c.sum())
    if n < 2:
        return np.nan
    choose2 = lambda a: np.sum(a*(a-1.)/2.)
    nij = choose2(c)
    a = choose2(c.sum(axis=1))
    b = choose2(c.sum(axis=0))
    expected = a*b/(n*(n-1.)/2.)
    denominator = .5*(a+b)-expected
    return float((nij-expected)/denominator) if denominator > 0 else 1.0

all_results = []
for sid, d in D.items():
    start = time.perf_counter()
    C = interval_costs(d)
    spans, objective = gene_specific_dp(C, d['K'], MIN_BINS_PER_SEGMENT)
    del C
    d['spans_by_gene'] = spans
    votes = d['votes']
    prefix = np.vstack([np.zeros((1, d['K']), dtype=np.int64), np.cumsum(votes, axis=0)])
    rows = []
    for g, gene in enumerate(d['names']):
        sp = spans[g]
        contingency = np.stack([prefix[j] - prefix[i] for i, j in sp])
        score = ari_from_contingency(contingency)
        changes = [int(j) for i, j in sp[:-1]]
        row = dict(
            slice_id=sid, gw=d['gw'], sample=d['sample'], region=d['region'],
            gene=str(gene), K=d['K'], n_manual_cells=d['n_labeled_cells'],
            ARI=score, poisson_objective=float(objective[g]),
            detection_fraction=float(d['detection'].get(str(gene), np.nan)),
            breakpoint_bins=json.dumps(changes),
            breakpoint_native_z=json.dumps([float(d['edges'][j]) for j in changes])
        )
        rows.append(row)
    ranked = pd.DataFrame(rows).sort_values(
        ['ARI', 'detection_fraction', 'gene'],
        ascending=[False, False, True]
    )
    ranked.to_csv(OUT/f'section_{sid:02d}_gene_specific_ARI.csv', index=False)
    all_results.extend(rows)
    top = ranked.iloc[0]
    print(f'  {d["gw"]} sid {sid}: BEST {top.gene}, ARI={top.ARI:.4f}, '
          f'{len(ranked)} genes, {time.perf_counter()-start:.1f}s')

all_df = pd.DataFrame(all_results).sort_values(
    ['ARI', 'detection_fraction', 'gene'],
    ascending=[False, False, True]
).reset_index(drop=True)
if len(all_df) == 0:
    raise RuntimeError('No ARI results')
all_df.insert(0, 'rank', np.arange(1, len(all_df)+1))
all_df.to_csv(OUT/'ALL_gene_tissue_ARI_ranked.csv', index=False)
summary = all_df.sort_values(['ARI', 'rank'], ascending=[False, True]).drop_duplicates('slice_id')
summary = summary.sort_values(['gw', 'slice_id'])
summary.to_csv(OUT/'BEST_gene_per_tissue_ARI.csv', index=False)

winner = all_df.iloc[0]
d = D[int(winner.slice_id)]
gene = str(winner.gene)
gene_i = int(np.flatnonzero(d['names'] == gene)[0])

print('\n' + '='*72)
print(f'GLOBAL HIGHEST (exploratory): {gene} in {winner.gw} '
      f'{winner["sample"]}/{winner["region"]}, sid={winner.slice_id}, '
      f'ARI={winner.ARI:.4f}, manual K={winner.K}')
print('Top 15 gene–tissue pairs:')
print(all_df[['rank','gw','slice_id','gene','K','ARI','detection_fraction']].head(15).to_string(index=False))
print('='*72)

## Render Figure 5B

In [ ]:
# Plot only. Run AFTER the Poisson/ARI analysis in the same kernel.
# Uses existing D, all_df, etc.; rebuilds the draw_profile function with fixed strips.
# Does not recompute ARI or retrain models.
from pathlib import Path
import numpy as np
from scipy.ndimage import gaussian_filter1d
from matplotlib import pyplot as plt
from matplotlib import patches as mpatches
from matplotlib import lines as mlines
from matplotlib import ticker as mticker
for key in ('D','all_df','SECTIONS','COLORS','INK','GRID','SCALE','DPI','OUT','fit_poisson_interval'):
    if key not in globals():
        raise RuntimeError(f'Missing {key}; run the original full analysis cell first.')
OUT=Path(OUT)
OUT.mkdir(parents=True,exist_ok=True)

# -------------------- 5. Annotated zone display --------------------
def manual_partition(votes):
    B, K = votes.shape
    sm = gaussian_filter1d(votes.astype(float), sigma=1., axis=0, mode='constant')
    probs = (sm + .5) / (sm.sum(1, keepdims=True) + .5*K)
    prefix = np.vstack([np.zeros((1, K)), np.cumsum(np.log(probs), axis=0)])
    dp = np.full((K+1, B+1), -np.inf)
    back = np.full((K+1, B+1), -1, int)
    dp[0, 0] = 0.
    for k in range(1, K+1):
        for j in range(k, B-(K-k)+1):
            ii = np.arange(k-1, j)
            val = dp[k-1, ii] + prefix[j, k-1] - prefix[ii, k-1]
            q = int(np.argmax(val))
            dp[k, j] = val[q]
            back[k, j] = ii[q]
    seg = []
    j = B
    for k in range(K, 0, -1):
        i = int(back[k, j])
        seg.append((i, j))
        j = i
    return seg[::-1]

def tissue_number_from_sid(gw, sid):
    return 1 + [s[0] for s in SECTIONS if s[1] == gw].index(int(sid))

def draw_profile(ax, row):
    d = D[int(row.slice_id)]
    gene = str(row.gene)
    gene_i = int(np.flatnonzero(d['names'] == gene)[0])
    spans = d['spans_by_gene'][gene_i]
    B = len(d['x'])
    x = d['x']
    W = d['W']
    Y = d['Y'][:, gene_i]
    edges = d['edges']
    xmin, xmax = edges[[0, -1]]
    normalize = lambda zz: (np.asarray(zz) - xmin) / (xmax - xmin)
    ref_spans = manual_partition(d['votes'])

    # Axes style
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines['left'].set_linewidth(0.72)
    ax.spines['bottom'].set_linewidth(0.72)
    ax.spines['left'].set_color(INK)
    ax.spines['bottom'].set_color(INK)
    ax.grid(axis='y', color=GRID, lw=0.52, alpha=0.34)
    ax.set_axisbelow(True)
    ax.tick_params(length=2.7, width=0.68, direction='out', pad=2.0, colors=INK)

    log_obs = np.full(B, np.nan)
    pos = Y > 0
    log_obs[pos] = np.log(SCALE * Y[pos] / W[pos])

    ax.scatter(
        normalize(x[pos]),
        log_obs[pos],
        s=11,
        color='#7A828C',
        alpha=0.64,
        linewidths=0,
        zorder=4,
        label='Observed'
    )

    fit_y = []
    for k, (i, j) in enumerate(spans):
        _, aa, bb = fit_poisson_interval(Y[i:j, None], W[i:j], x[i:j], return_coeff=True)
        xq = np.linspace(x[i], x[j-1], 90)
        yq = np.log(SCALE) + aa[0] + bb[0] * xq
        if np.isfinite(yq).all():
            fit_y.append(yq)
            ax.plot(
                normalize(xq),
                yq,
                lw=1.85,
                color=COLORS[d['zone_ids'][k]],
                zorder=5
            )

    for i, j in spans[:-1]:
        ax.axvline(
            normalize(edges[j]),
            lw=0.95,
            color=INK,
            ls=(0, (3.2, 2.1)),
            zorder=2
        )

    good_observed = log_obs[np.isfinite(log_obs)]
    if not len(good_observed) and not fit_y:
        raise RuntimeError(f'{gene}: no finite log-expression values to plot')
    limits = np.concatenate(([good_observed] if len(good_observed) else []) + fit_y)
    lo, hi = limits.min(), limits.max()
    pad = max(.25, .09 * max(.1, hi - lo))
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_xlim(0, 1)

    ax.xaxis.set_major_locator(mticker.MaxNLocator(5))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(4))

    ax.set_ylabel('ln(transcripts / 10$^4$ UMIs)', fontsize=9.2, labelpad=1.6)

    tissue_number = tissue_number_from_sid(d['gw'], int(row.slice_id))
    # Split the long heading into two centered rows so ARI is never clipped.
    ax.set_title(
        f'{gene} · {d["gw"]} Tissue {tissue_number}\nARI = {float(row.ARI):.3f}',
        pad=3.5,
        fontsize=9.2,
        fontweight='bold',
        color=INK,
        linespacing=1.08,
    )

    for (i, j), lab in zip(ref_spans, d['zone_ids']):
        left, right = normalize(edges[[i, j]])
        ax.add_patch(mpatches.Rectangle(
            (left, -0.280), right-left, 0.055,
            transform=ax.get_xaxis_transform(),
            color=COLORS[lab], lw=0, clip_on=False, zorder=6
        ))

# -------------------- 6. Stacked top-2 panel in publication style --------------------
top_n = min(2, len(all_df))
top_rows = all_df.head(top_n).copy()

# Keep the same physical size
FIG_W = 2.70
FIG_H = 4.38

fig, axs = plt.subplots(top_n, 1, figsize=(FIG_W, FIG_H), facecolor='white')
if top_n == 1:
    axs = [axs]

assert top_n == 2, 'This publication layout expects the top two ranked pairs'

# Slightly cleaner spacing in the final style guide
panel_positions = [
    (0.27, 0.662, 0.675, 0.240),
    (0.27, 0.256, 0.675, 0.258),  # two-line titles clear the upper strip
]
for ax, position in zip(axs, panel_positions):
    ax.set_position(position)

for panel_i, (ax, row) in enumerate(zip(axs, top_rows.itertuples(index=False))):
    draw_profile(ax, row)
    ax.yaxis.set_label_coords(-0.17, 0.36)
    if panel_i == 0:
        ax.set_xlabel('')
        ax.tick_params(axis='x', labelbottom=False)
    else:
        ax.set_xlabel('Common coordinate', fontsize=9.6, labelpad=8.5)
        ax.xaxis.set_label_coords(0.5, -0.35)  # below the colored strip

# Moderately enlarged footer legend (9.6 pt) in two centered rows.
# The 2.70-inch figure is too narrow for three readable items on one row.
# Both rows sit below the shared x-label, without enlarging the figure.
legend_handles = [
    mlines.Line2D([0], [0], marker='o', lw=0, ms=5.8,
                  color='#7A828C', label='Observed'),
    mlines.Line2D([0], [0], lw=2.15, color=INK, label='Poisson fit'),
    mlines.Line2D([0], [0], lw=1.35, color=INK,
                  ls=(0, (3.2, 2.1)), label='Inferred break'),
]
legend_top = fig.legend(
    handles=legend_handles[:2],
    ncol=2,
    loc='lower center',
    bbox_to_anchor=(0.50, 0.290 / FIG_H),
    fontsize=9.6,
    frameon=False,
    columnspacing=0.23,
    handlelength=0.86,
    handletextpad=0.17,
    borderaxespad=0.0,
)
legend_bottom = fig.legend(
    handles=legend_handles[2:],
    ncol=1,
    loc='lower center',
    bbox_to_anchor=(0.50, 0.035 / FIG_H),
    fontsize=9.6,
    frameon=False,
    handlelength=0.86,
    handletextpad=0.17,
    borderaxespad=0.0,
)

# SAVE FIRST: no layout-check assertion can prevent the figure from being written.
# Exact canvas size is preserved (2.70 x 4.38 inches); no tight bbox crop.
OUT = Path(OUT)
OUT.mkdir(parents=True, exist_ok=True)
stem = OUT / 'FIG_top2_gene_tissue_ARI_width270_h438_PUBLICATION_TWOLINE_ARI_FINAL'
written = []
for ext in ('png', 'pdf', 'svg'):
    target = stem.with_suffix('.' + ext)
    fig.savefig(
        target,
        dpi=DPI if ext == 'png' else None,
        facecolor='white', bbox_inches=None, pad_inches=0
    )
    if not target.is_file() or target.stat().st_size == 0:
        raise OSError(f'Figure export did not create a nonempty file: {target}')
    written.append(target)
    print(f'SAVED [{target.stat().st_size:,} bytes]: {target.resolve()}', flush=True)

# NONFATAL LAYOUT AUDIT: report any warnings AFTER successful exports.
# Matplotlib/font renderers vary across Skipjack and PowerPoint systems.
try:
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    inch = fig.dpi_scale_trans.inverted()
    l1 = legend_top.get_window_extent(renderer).transformed(inch)
    l2 = legend_bottom.get_window_extent(renderer).transformed(inch)
    xlabel_box = axs[-1].xaxis.label.get_window_extent(renderer).transformed(inch)
    if not (l1.x0 >= 0 and l1.x1 <= FIG_W and l2.x0 >= 0 and l2.x1 <= FIG_W):
        print('LAYOUT NOTE: legend extends beyond nominal width:', l1.bounds, l2.bounds)
    if l2.y1 > l1.y0:
        print('LAYOUT NOTE: legend rows may touch:', l2.bounds, l1.bounds)
    if l1.y1 > xlabel_box.y0:
        print('LAYOUT NOTE: legend may touch x-axis label:', l1.bounds, xlabel_box.bounds)
    for ax in axs:
        box = ax.title.get_window_extent(renderer).transformed(inch)
        if box.x0 < 0 or box.x1 > FIG_W or box.y1 > FIG_H:
            print('LAYOUT NOTE: title extends beyond canvas:', box.bounds)
    print('Footer clearance (in):', round(float(xlabel_box.y0-l1.y1), 4))
except Exception as exc:
    print('LAYOUT NOTE: post-save diagnostics unavailable:', repr(exc))

print('Figure exported successfully to:', OUT.resolve())

plt.show()
plt.close(fig)

print()
print('Top 2 overall gene–tissue pairs selected for stacked plotting:')
print(top_rows[['rank','gw','slice_id','gene','K','ARI','detection_fraction']].to_string(index=False))